In [1]:
!pip install --upgrade keras

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 6.4 MB/s eta 0:00:00
  Attempting uninstall: keras
    Found existing installation: keras 2.15.0
    Uninstalling keras-2.15.0:
      Successfully uninstalled keras-2.15.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.15.0 requires keras<2.16,>=2.15.0, but you have keras 3.0.5 which is incompatible.


In [2]:
import keras

print(keras.__version__)

3.0.5


In [3]:
import numpy as np
import tensorflow as tf
from keras.datasets import cifar10
from keras import Sequential
from keras.layers import Input, Dense, Dropout, Flatten, Conv2D, MaxPooling2D
from keras.layers import Rescaling, RandomFlip, RandomRotation, RandomZoom
from keras.utils import to_categorical

np.random.seed(10)  # 指定亂數種子
# 載入資料集
(X_train, y_train), (X_test, y_test) = cifar10.load_data()
# One-hot編碼
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)
# 建立圖片增強層的 Sequential模型
data_augmentation = Sequential(
    [
        RandomFlip("horizontal"),
        RandomRotation(0.1),
        RandomZoom(0.1)
    ]
)
# 建立 tf.data.Dataset 套用圖片增強層
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.batch(128).map(lambda x, y: (data_augmentation(x), y))
# 定義模型
model = Sequential()
model.add(Input(shape=X_train.shape[1:]))
model.add(Rescaling(scale=1./255))      # 加上圖片預處理層
model.add(Conv2D(32, kernel_size=(3, 3), padding="same",
                 activation="relu"))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Conv2D(64, kernel_size=(3, 3), padding="same",
                 activation="relu"))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.5))
model.add(Flatten())
model.add(Dense(256, activation="relu"))
model.add(Dropout(0.5))
model.add(Dense(10, activation="softmax"))
model.summary()   # 顯示模型摘要資訊
print("--------------------------")
# 編譯模型
model.compile(loss="categorical_crossentropy", optimizer="adam",
              metrics=["accuracy"])
# 訓練模型
model.fit(train_ds, epochs=100, verbose=2,
          validation_data=(X_test, y_test))
# 評估模型
print("\nTesting ...")
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print("測試資料集的準確度 = {:.2f}".format(accuracy))

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)                │ (None, 32, 32, 3)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d (Conv2D)                      │ (None, 32, 32, 32)          │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 16, 16, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 16, 16, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 8, 8, 64)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 8, 8, 64)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 4096)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │       1,048,832 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 10)                  │           2,570 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,070,794 (4.08 MB)

 Trainable params: 1,070,794 (4.08 MB)

 Non-trainable params: 0 (0.00 B)

--------------------------
Epoch 1/100
391/391 - 27s - 70ms/step - accuracy: 0.3634 - loss: 1.7479 - val_accuracy: 0.4808 - val_loss: 1.4357
Epoch 2/100
391/391 - 30s - 76ms/step - accuracy: 0.4613 - loss: 1.4996 - val_accuracy: 0.5339 - val_loss: 1.2890
Epoch 3/100
391/391 - 20s - 52ms/step - accuracy: 0.4913 - loss: 1.4167 - val_accuracy: 0.5571 - val_loss: 1.2403
Epoch 4/100
391/391 - 21s - 54ms/step - accuracy: 0.5163 - loss: 1.3556 - val_accuracy: 0.5910 - val_loss: 1.1533
Epoch 5/100
391/391 - 20s - 52ms/step - accuracy: 0.5314 - loss: 1.3164 - val_accuracy: 0.5926 - val_loss: 1.1409
Epoch 6/100
391/391 - 14s - 36ms/step - accuracy: 0.5419 - loss: 1.2857 - val_accuracy: 0.6158 - val_loss: 1.0765
Epoch 7/100
391/391 - 14s - 36ms/step - accuracy: 0.5509 - loss: 1.2600 - val_accuracy: 0.5976 - val_loss: 1.1251
Epoch 8/100
391/391 - 21s - 53ms/step - accuracy: 0.5613 - loss: 1.2339 - val_accuracy: 0.6307 - val_loss: 1.0419
Epoch 9/100
391/391 - 14s - 37ms/step - accuracy: 0.5722 - lo